In [47]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
model=ChatOpenAI()

In [48]:
#flow is that first post generation then evaluation adn if approved then end otherwise sent to optimiser for feedback and then again evalutated 

In [49]:
class post(TypedDict):
    topic:str
    gen_post:str
    feedback_result:Literal["approved","needs improvement"]
    feedback_txt:str
    final_post:str
    
class eval(BaseModel):
    feedback_result:Literal["approved","needs improvement"]
    feedback_txt:str

eval_model=model.with_structured_output(eval)


In [ ]:
def generate(state:post)->post:
    prompt=f"Generate a social media post on the topic: {state['topic']} and use very less tokens"
    response=model.invoke(prompt)
    generated_post=response.content
    state['gen_post']=generated_post
    return {'gen_post':generated_post}

def evaluate(state:post)->post:
    prompt=f"Evaluate the following post: {state['gen_post']} and provide feedback. If it is good, return 'approved' else return 'needs improvement' and provide feedback text."
    response=eval_model.invoke(prompt)
    state['feedback_result']=response.feedback_result
    state['feedback_txt']=response.feedback_txt
    return {'feedback_result':response.feedback_result,'feedback_txt':response.feedback_txt}

def optimise(state:post)->post:
    prompt=f"Optimise the following post: {state['gen_post']} based on the feedback: {state['feedback_txt']}. Return the improved post."
    response=model.invoke(prompt)
    improved_post=response.content
    state['gen_post']=improved_post
    return {'gen_post':improved_post}


def check(state:post)
    if state['feedback_result']=="approved":
        state['final_post']=state['gen_post']
        return END
    else:
        return "Optimise Post"

In [51]:
graph=StateGraph(post)

graph.add_node("Generate Post",generate)
graph.add_node("Evaluate Post",evaluate)
graph.add_node("Optimise Post",optimise)

graph.add_edge(START,"Generate Post")
graph.add_edge("Generate Post","Evaluate Post")
graph.add_conditional_edges("Evaluate Post",check,{END: END,"Optimise Post":"Optimise Post"})
graph.add_edge("Optimise Post","Evaluate Post")


workflow=graph.compile()




In [52]:
workflow.invoke({"topic":"The importance of AI in modern society"})

{'topic': 'The importance of AI in modern society',
 'gen_post': 'AI is revolutionizing our world, from personalized recommendations to autonomous vehicles. Its impact on modern society is undeniable. #AI #techrevolution',
 'feedback_result': 'approved',
 'feedback_txt': 'The evaluation is approved. The post is well-written and effectively highlights the impact of AI on modern society. It effectively conveys the importance of AI in revolutionizing different aspects of our world.'}